In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from stable_baselines3 import A2C
from trading_env import TradingEnv

In [2]:
model = A2C.load("models/a2c_reward_v3")

In [3]:
stocks = ["AAPL", "AMZN", "JNJ", "JPM", "MSFT", "TSLA"]

test_dataframes = []

for stock in stocks:
    df = pd.read_csv(f"data/train/{stock}.csv")
    df["Date"] = pd.to_datetime(df["Date"])
    test_dataframes.append(df)

print(f"Loaded {len(test_dataframes)} test datasets.")

Loaded 6 test datasets.


In [4]:
test_env = TradingEnv(test_dataframes)

## Evaluation

In [5]:
obs, info = test_env.reset()

done = False

portfolio_history = []
reward_history = []
action_history = []
price_history = []
date_history = []

In [6]:
while not done:

    # Agent predicts action
    action, _ = model.predict(obs, deterministic=True)

    # Store action
    action_history.append(int(action))

    # Current market information
    price_history.append(test_env.df.loc[test_env.current_step, "Close"])
    date_history.append(test_env.df.loc[test_env.current_step, "Date"])

    # Take action
    obs, reward, terminated, truncated, info = test_env.step(action)

    # Store results
    reward_history.append(reward)
    portfolio_history.append(test_env.portfolio_value)

    done = terminated or truncated

In [7]:
print(f"Total Trading Days : {len(portfolio_history)}")
print(f"Final Portfolio    : ${portfolio_history[-1]:.2f}")
print(f"Total Reward       : {sum(reward_history):.2f}")

Total Trading Days : 2195
Final Portfolio    : $10000.00
Total Reward       : 0.00


In [8]:
import numpy as np

actions, counts = np.unique(action_history, return_counts=True)

action_names = {
    0: "Hold",
    1: "Buy",
    2: "Sell"
}

for action, count in zip(actions, counts):
    print(f"{action_names[action]} : {count}")

Hold : 2195


In [9]:
obs, info = test_env.reset()

for i in range(10):
    action, _ = model.predict(obs, deterministic=True)
    print(f"Step {i}: Action = {action}")

    obs, reward, terminated, truncated, info = test_env.step(action)

    if terminated or truncated:
        break

Step 0: Action = 0
Step 1: Action = 0
Step 2: Action = 0
Step 3: Action = 0
Step 4: Action = 0
Step 5: Action = 0
Step 6: Action = 0
Step 7: Action = 0
Step 8: Action = 0
Step 9: Action = 0
